# Chaos-Economy: Multi-Agent VSR-Env GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manan-tech/Chaos-Economy/blob/main/vsr_env_kaggle.ipynb)

Welcome to the **Chaos-Economy** project! This notebook demonstrates how to train and evaluate a Multi-Agent Volatility Surface Response Environment (VSR-Env) using **Group Relative Policy Optimization (GRPO)** via Unsloth and TRL.

In this environment, multiple LLM agents (Traders, a Market Maker, and an SEC Regulator) interact with an options market. We use the `meta-llama/Llama-3.2-1B-Instruct` model and train it to navigate the complexities of financial derivatives, emergent collusion, and regulatory enforcement.

## 1. Environment Setup
We first install the necessary dependencies including Unsloth, TRL, and clone the Chaos-Economy repository.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.16.0" peft accelerate bitsandbytes
!git clone https://github.com/manan-tech/Chaos-Economy.git
%cd Chaos-Economy
!pip install -e .

## 2. Train the Multi-Agent Pipeline

We train the agents across 4 distinct narrative phases:
1. **Discovery** (Learn the mechanics)
2. **Exploitation** (Learn to profit)
3. **Collusion** (Emergent coordinated attacks)
4. **Adaptation** (Market Maker and SEC react)

*(Note: For demonstration purposes, this runs a shortened training loop. In our full run, we used 250 steps over multiple epochs.)*

In [ ]:
# Train using the Llama-3.2-1B-Instruct model
!python train_multi_agent_pipeline.py \
    --base_model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --num_episodes 1 \
    --episode_length 10 \
    --num_epochs 1 \
    --max_steps 40 \
    --learning_rate 5e-5 \
    --output_dir ./multi_agent_checkpoints \
    --use_wandb False

## 3. Evaluate the Model (Base vs. Trained LoRA)

Now we evaluate the unified model against the base Llama-3.2-1B-Instruct model. The evaluation script plays 30 steps of the market simulation and outputs two replay files:
- `unified_lora_replay.json` (Our RL-trained model)
- `unified_baseline_replay.json` (The base model without LoRA adapter)

You will see logs of trades, Market Maker spread adjustments, and SEC enforcement actions.

In [ ]:
!python test_unified_kaggle.py \
    --base_model unsloth/Llama-3.2-1B-Instruct-bnb-4bit \
    --num_episodes 1 \
    --num_steps 30 \
    --seed 42

## 4. View Replay Results

You can now inspect the generated replay JSONs or use the visualization tools provided in the repository to replay the market dynamics.

In [ ]:
import json

try:
    with open("unified_lora_replay.json", "r") as f:
        data = json.load(f)
        print("Trained LoRA Final Rewards:", data["final_rewards"])
        print("\nTotal SEC Fines Issued:", sum(step.get("sec_fines", 0) for step in data["steps"]))
except FileNotFoundError:
    print("Replay file not found. Ensure the evaluation script completed successfully.")